In [0]:
%pip install /Volumes/openalex/default/libraries/openalex_dlt_utils-0.2.1-py3-none-any.whl

In [ ]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

from openalex.utils.environment import *

# ============================================================================
# CreateSources, post-cutover (oxjob #548)
#
# The sources registry lives in the openalex-sources Heroku Postgres and is
# maintained by that app's feed / merge / curation jobs. This pipeline no
# longer BUILDS sources -- it materializes a daily, run-consistent snapshot of
# the federated registry (openalex_sources.public.sources) in the legacy shape,
# so the four downstream consumers (CreateSourcesApi, CreateLocationsWithSources,
# CreateWorksBase, CreateInstitutionsApi) run unchanged.
#
# Legacy-shape contract (registry -> this table):
#   issn     = registry issn_l (ISSN-L; for a few recent mints not in issns)
#   webpage  = registry homepage_url
#   issns    = COALESCE(issns, array())  -- ISSN-less rows: [] not NULL
#   apc_prices / societies / alternate_titles / datacite_ids: JSONB -> arrays
#   created_date / updated_date: cast to string (legacy column types)
#   merged sources are INCLUDED as redirect rows (merge_into_id set);
#   consumers needing active-only filter merge_into_id IS NULL.
#
# One snapshot scan/day keeps end2end off the federated JDBC path (no repeated
# single-stream pulls, no mid-run mutation hazard from the app's triggers).
# ============================================================================

APC_SCHEMA = ArrayType(StructType([
    StructField("price", IntegerType(), True),
    StructField("currency", StringType(), True),
]))

SOCIETIES_SCHEMA = ArrayType(StructType([
    StructField("url", StringType(), True),
    StructField("organization", StringType(), True),
]))


@dlt.table(
    name="sources",
    comment=f"Snapshot of the openalex-sources Postgres registry (legacy shape) in {ENV.upper()}"
)
def sources():
    return (
        spark.table("openalex_sources.public.sources")
        .select(
            col("id"),
            col("endpoint_id"),
            col("display_name"),
            col("issn_l").alias("issn"),
            col("publisher"),
            col("homepage_url").alias("webpage"),
            col("is_oa"),
            col("type"),
            from_json(col("apc_prices"), APC_SCHEMA).alias("apc_prices"),
            col("is_society_journal"),
            from_json(col("societies"), SOCIETIES_SCHEMA).alias("societies"),
            col("apc_usd"),
            col("fatcat_id"),
            col("wikidata_id"),
            col("crossref_id"),
            col("country"),
            col("country_code"),
            from_json(col("alternate_titles"), ArrayType(StringType())).alias("alternate_titles"),
            col("publisher_id"),
            col("institution_id"),
            col("is_core"),
            col("merge_into_id"),
            col("merge_into_date"),
            col("updated_date").cast("string").alias("updated_date"),
            col("created_date").cast("string").alias("created_date"),
            col("display_name_before_override"),
            col("override_timestamp"),
            col("datacite_id"),
            expr("coalesce(issns, array())").alias("issns"),
            col("is_in_doaj"),
            col("is_in_doaj_start_year"),
            col("doaj_license"),
            col("is_in_scielo"),
            col("is_ojs"),
            col("is_oa_high_oa_rate"),
            col("high_oa_rate_start_year"),
            col("is_fully_open_in_jstage"),
            col("sample_pmh_record"),
            expr("coalesce(from_json(datacite_ids, 'array<string>'), array())").alias("datacite_ids"),
            col("is_preprint_repository"),
        )
    )
